In [1]:
# ──────────────────────────────────────────────────────────────────────────────
# Task 3 : Distributed Machine Learning Models
# Setup
# ──────────────────────────────────────────────────────────────────────────────

from google.colab import drive
drive.mount('/content/drive')

import os
import sys
import time

os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"
os.environ["PYSPARK_PYTHON"] = sys.executable

from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("7006SCN_HMDA_Task3")
    .master("local[*]")
    .config("spark.driver.memory","10g")
    .config("spark.sql.shuffle.partitions","200")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")

print(f"Spark Version : {spark.version}")

# ──────────────────────────────────────────────────────────────────────────────
# Required Libraries
# ──────────────────────────────────────────────────────────────────────────────

from pyspark.ml import PipelineModel

from pyspark.ml.classification import (
    LogisticRegression,
    DecisionTreeClassifier,
    RandomForestClassifier,
    GBTClassifier
)

from pyspark.ml.evaluation import (
    BinaryClassificationEvaluator,
    MulticlassClassificationEvaluator
)

from pyspark.ml.tuning import (
    CrossValidator,
    ParamGridBuilder
)

from pyspark.storagelevel import StorageLevel

# ──────────────────────────────────────────────────────────────────────────────
# Project Directories
# ──────────────────────────────────────────────────────────────────────────────

BASE = "/content/drive/MyDrive/7006SCN"

PROCESSED = f"{BASE}/data/processed"

MODELS = f"{BASE}/models"

OUTPUT = f"{BASE}/outputs"

print("Project directories configured ✓")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Spark Version : 4.0.3
Project directories configured ✓


In [2]:
# ──────────────────────────────────────────────────────────────────────────────
# Load HMDA Clean Dataset
# ──────────────────────────────────────────────────────────────────────────────

df_clean = spark.read.parquet(
    f"{PROCESSED}/hmda_clean.parquet"
)

print("="*80)
print("HMDA CLEAN DATASET LOADED")
print("="*80)

print(f"Rows    : {df_clean.count():,}")
print(f"Columns : {len(df_clean.columns)}")

HMDA CLEAN DATASET LOADED
Rows    : 11,564,178
Columns : 123


In [3]:
# ──────────────────────────────────────────────────────────────────────────────
# Load Spark Preprocessing Pipeline
# ──────────────────────────────────────────────────────────────────────────────

pipeline_model = PipelineModel.load(
    f"{MODELS}/preprocessing_pipeline"
)

print("Preprocessing Pipeline Loaded ✓")
# ──────────────────────────────────────────────────────────────────────────────
# Generate Feature Vector
# ──────────────────────────────────────────────────────────────────────────────

df_model = pipeline_model.transform(df_clean)

print("="*80)
print("FEATURE ENGINEERING COMPLETE")
print("="*80)

df_model.select(
    "loan_approval",
    "features"
).show(5, truncate=False)
# ──────────────────────────────────────────────────────────────────────────────
# Train-Test Split
# ──────────────────────────────────────────────────────────────────────────────

train, test = df_model.randomSplit(
    [0.8,0.2],
    seed=7006
)

# -------------------------------------------------------------------------
# Representative Sampling (10%)
# -------------------------------------------------------------------------

train = (
    train.sample(
        withReplacement=False,
        fraction=0.10,
        seed=7006
    )
    .persist(StorageLevel.MEMORY_AND_DISK)
)

test = (
    test.sample(
        withReplacement=False,
        fraction=0.10,
        seed=7006
    )
    .persist(StorageLevel.MEMORY_AND_DISK)
)

print()

print("="*80)
print("Sampled Dataset")
print("="*80)

print(f"Training : {train.count():,}")

print(f"Testing  : {test.count():,}")


# ──────────────────────────────────────────────────────────────────────────────
# Evaluation Metrics
# ──────────────────────────────────────────────────────────────────────────────

ev_acc = MulticlassClassificationEvaluator(
    labelCol="loan_approval",
    predictionCol="prediction",
    metricName="accuracy"
)

ev_f1 = MulticlassClassificationEvaluator(
    labelCol="loan_approval",
    predictionCol="prediction",
    metricName="f1"
)

ev_auc = BinaryClassificationEvaluator(
    labelCol="loan_approval",
    rawPredictionCol="rawPrediction",
    metricName="areaUnderROC"
)

results = []

print("Evaluators Ready ✓")
# ──────────────────────────────────────────────────────────────────────────────
# Commit 1 (LR): Baseline Logistic Regression
# feat(task3/lr): baseline Logistic Regression model
#
# Justification:
# Logistic Regression is selected as a strong interpretable baseline classifier
# for predicting mortgage loan approval. The model estimates approval
# probabilities from applicant, loan and property characteristics and serves
# as a benchmark before evaluating more complex tree-based ensemble models.
# ──────────────────────────────────────────────────────────────────────────────

from pyspark.ml.classification import LogisticRegression

print("=" * 80)
print("COMMIT 1 : BASELINE LOGISTIC REGRESSION")
print("=" * 80)

# -----------------------------------------------------------------------------
# Create Logistic Regression Model
# -----------------------------------------------------------------------------

lr = LogisticRegression(
    featuresCol="features",
    labelCol="loan_approval",
    predictionCol="prediction",
    probabilityCol="probability",
    rawPredictionCol="rawPrediction",
    maxIter=20,
    regParam=0.01,
    elasticNetParam=0.0,
    standardization=True
)

print("Training Logistic Regression Model...")

start = time.time()

lr_model = lr.fit(train)

training_time = time.time() - start

print(f"Training completed in {training_time:.2f} seconds")

# -----------------------------------------------------------------------------
# Generate Predictions
# -----------------------------------------------------------------------------

pred_lr = lr_model.transform(test)

# -----------------------------------------------------------------------------
# Evaluate Model
# -----------------------------------------------------------------------------

accuracy = ev_acc.evaluate(pred_lr)
f1_score = ev_f1.evaluate(pred_lr)
roc_auc = ev_auc.evaluate(pred_lr)

print("\n")
print("=" * 80)
print("BASELINE LOGISTIC REGRESSION RESULTS")
print("=" * 80)

print(f"Accuracy      : {accuracy:.4f}")
print(f"F1-Score      : {f1_score:.4f}")
print(f"ROC-AUC       : {roc_auc:.4f}")
print(f"Training Time : {training_time:.2f} seconds")

# -----------------------------------------------------------------------------
# Save Baseline Model
# -----------------------------------------------------------------------------

lr_model.write() \
    .overwrite() \
    .save(f"{MODELS}/lr_baseline")

print("Baseline Logistic Regression model saved ✓")

# -----------------------------------------------------------------------------
# Store Results
# -----------------------------------------------------------------------------

results.append({
    "Model": "Logistic Regression",
    "Version": "Baseline",
    "Accuracy": round(accuracy, 4),
    "F1-Score": round(f1_score, 4),
    "ROC-AUC": round(roc_auc, 4),
    "Training Time (s)": round(training_time, 2)
})

print("=" * 80)
print("COMMIT 1 COMPLETED")
print("=" * 80)

Preprocessing Pipeline Loaded ✓
